In [1]:
import os
os.chdir('..')

import warnings
warnings.filterwarnings('ignore')

In [2]:
from dotenv import load_dotenv
load_dotenv()

import mlflow
import pandas as pd

import bert_score
from mlflow.entities import Feedback
from mlflow.genai import(
    evaluate,
    scorer,
    load_prompt,
    get_scorer,
)

from mlflow.genai.scorers import Guidelines

from mlflow.genai.datasets import get_dataset
from src.utils.mlflow_utils import load_model, load_models_by_groups

python-dotenv could not parse statement starting at line 17


In [3]:
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACING_URI'])
mlflow.set_experiment("dialogue_summarization")
mlflow.langchain.autolog()

experiment_id = mlflow.get_experiment_by_name("dialogue_summarization").experiment_id

In [4]:
@scorer
def bert_scorer(inputs, outputs, expectations):
    op_summary = outputs
    gold_summary = expectations["summary"]

    try:
        bertscore = bert_score.score(
            cands=[op_summary],
            refs=[gold_summary],
            model_type="roberta-base"
        )
    except:
        return [Feedback(value=None, rationale="Error evaluating metric."),]*3

    metric_names = ["P", "R", "F"]
    
    feedbacks = [
        Feedback(value=float(value), name=metric_names[i])
        for i, value in enumerate(bertscore)
    ]

    return feedbacks

In [8]:
models = load_models_by_groups(groups=["local", "openai"])

for llm in models:
    print(llm["id"])
    # print(llm['model'].invoke("What is capital of Norway ?").content)

llama3.1:8b
gemma3:4b
gpt-5.2


In [9]:
prompt_id = "prompts:/dialog_summarization/4"
prompt = load_prompt(prompt_id)

dataset = get_dataset("dialogue_summarization_sampled")

# judge_named_person = get_scorer(name="Named Person", experiment_id=experiment_id)

In [10]:
guidelines = [
    Guidelines(
        name="privacy_aware",
        guidelines=[
            "The response should not leak any personal names."
        ],
        model="gateway:/gpt-5-2"
    ),
    Guidelines(
        name="respond_in_english",
        guidelines=[
            "The response should always be in english language."
        ],
        model="gateway:/gpt-5-2"
    )
]


for model in models:
    llm = model['model']

    with mlflow.start_run(
        run_name=f"Dialog Summary : {model["id"]}",
        description="Run description.",
        # tags={"env": "dev"},
        experiment_id=experiment_id,
    ):
        mlflow.log_params({
            "summarizer": model["id"],
            "privacy_chekcer": "gpt-5-2",
            "language_checker": "gpt-5-2",
            "prompt": prompt_id,
        })

        _model = mlflow.create_external_model(name=model["name"], model_type="llm")
        mlflow.set_active_model(model_id=_model.model_id)
        mlflow.log_model_params({
            "temperature": 0.8,
        }, model_id=_model.model_id)


        @mlflow.trace
        def predict_fn(**kargs):
            response = llm.invoke(prompt.format(**kargs))

            return response.content

        results = mlflow.genai.evaluate(
            data=dataset,
            predict_fn=predict_fn,
            scorers=[bert_scorer, *guidelines]
        )

        mlflow.log_metrics({
            "custom_scorer": results.result_df[["P/value", "R/value", "F/value"]].max(axis=1).mean()
        })

   

2026/03/20 14:16:33 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/03/20 14:16:33 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/03/20 14:16:33 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-f9c4ecd47ea743d79883c2415a887b5b
2026/03/20 14:16:33 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026/03/20 14:16:33 WARNING mlflow.tracing.fluent: Failed to start span ChatOpenAI: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7734.95it/s]
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-


2026/03/20 14:16:55 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/20 14:16:55 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
2026/03/20 14:16:55 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/03/20 14:16:55 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/03/20 14:16:55 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-509f1c595182405890031c42a5e4f052
2026/03/20 14:16:55 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026/03/20 14:16:55 WARNING mlflow.tracing.fluent: Failed to start span ChatOpenAI: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set lo

2026/03/20 14:17:15 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/20 14:17:15 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
2026/03/20 14:17:15 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/03/20 14:17:15 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/03/20 14:17:15 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-408359ad59a04329b738f091f6d9acc8
2026/03/20 14:17:15 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026/03/20 14:17:15 WARNING mlflow.tracing.fluent: Failed to start span ChatOpenAI: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set lo

2026/03/20 14:17:27 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/20 14:17:27 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
